In [37]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from crewai import Task, Agent, Crew

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from crewai import LLM
llm = LLM(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3
)

from tavily import TavilyClient
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [42]:
from crewai_tools import TavilySearchTool

search_tool = TavilySearchTool()

In [43]:
transportation_researcher = Agent(
    role="Transportation Researcher",
    goal="Recommend the best mode of transportation between {from_place} and {to_place}, based on {going_date} and {returning_date} and {transportation_budget}.",
    backstory="""You are a transportation expert with 20 years of experience. Your job is to research all available transportation options between between two places for a given going date and returning date, for a given number of persons. Analyze the distance, travel duration, schedules, convenience, and costs. Recommend the most suitable transportation option while staying within the user's transportation budget. Provide details such as timings, travel duration, estimated cost, and reasons for your recommendation.""",
    verbose=False,
    llm=llm,
    tools=[search_tool]
)

In [44]:
accommodation_researcher = Agent(
    role = "Accommodation Researcher",
    goal = "For a given budget, number of tourists persons and a period between two dates",
    backstory = """You are an experienced accommodation agent, who is the industry for the past 20 years and has contacts all over the world with all types of hotels, resorts, villas, homestays and all other tyoe of accommodations. If you are given duration of stay, destination, accommodation budget and the number of people, you are able to book the best accommodation and will also provide the reason of choosing a specific option over others""",
    llm = llm,
    tools = [search_tool],
    verbose = True
)

In [45]:
activities_researcher = Agent(
    role = "Activities Researcher",
    goal = "For a given number of days at a partcular place, recommend all the best activities to do there.",
    backstory = """You are an experienced activities planner, who has got more than 25 years of planning activities to do at a specific destination. You first assess the destinantio, number of people and the duration for which they will stay at that particular place. You thoroughly analyse which type of people are the in the group, look at the time of the year thay are going to visit the destination and every other relevant factor, and based on everything you have, you recommend the best activities to do in the place. You can recommend historical locations, food items to try, top places to visit and every other experience whcih can make the people remmeber that place for the rest of their lives, and tehy don't feel that they have wasted their time, energy and money in going to that place. Also you give reasons as to why you chose a specific activity.""",
    llm = llm,
    tools = [search_tool],
    verbose = True
)

In [46]:
itinerary_researcher = Agent(
    role = "Itinerary Researcher",
    goal = "Prepare a travl itinerary for a number of people, traveling to a place and wish to get the best experinced from all the activities they want to visit.",
    backstory = """You are a travel itinerary planner, who is in this profession for the past 30 years, and have drafted thousands of travel itineraries, and have played a sustantial role in making people get the best experience of the place they want to visit. You see the destination, duration of the trip and the number of person going and every other relevant detail you require so that you are able to draft a travel plan so that people could get the maximum out of a certain place.""",
    llm = llm,
    verbose = True
)

In [47]:
qa_researcher = Agent(
    role = "Quality Assurance Researcher",
    goal = "Check and cross verify all the data gathered before presenting it finally.",
    backstory = """You are an industry level quality asuurance researcher who have worked in many of the top firms in the world. You are an expert in checking all the facts and assumptions used during analysis and research and pointing out""",
    llm = llm,
    verbose = True
)

In [48]:
transportation_search = Task(
    description="""
    Research transportation options for a trip from {from_place} to {to_place}.The trip starts on {going_date} and ends on {returning_date}. The number of adults on the trip are {number_of_adults} and number of kids are {number_of_kids} on the trip, and the total transportation budget is {transportation_budget}.Consider all practical transportation options available for this route. Compare them on the basis of cost, travel duration, convenience, and overall suitability for the travelers.Recommend the option that provides the best balance between cost and convenience while remaining within the transportation budget.Include details for both the outbound and return journeys. Use available search tools to gather relevant and up-to-date information wherever necessary.
    """,
    expected_output="""
    Transportation Analysis

    1. Options Considered
       - List the transportation options evaluated.

    2. Recommended Transportation Option
       - Name of the recommended option.
       - Brief explanation of why it was selected.

    3. Outbound Journey ({from_place} to {to_place})
       - Mode of transportation
       - Estimated departure time
       - Estimated arrival time
       - Total travel duration
       - Estimated cost

    4. Return Journey ({to_place} → {from_place})
       - Mode of transportation
       - Estimated departure time
       - Estimated arrival time
       - Total travel duration
       - Estimated cost

    5. Cost Summary
       - Outbound cost
       - Return cost
       - Total estimated transportation cost

    6. Budget Assessment
       - Whether the recommendation fits within the transportation budget.

    7. Recommendation Rationale
       - Advantages of the selected option.
       - Comparison with other feasible alternatives.
    """,

    agent=transportation_researcher
)

In [49]:
accommodation_search = Task(
    description = ("""Recommend a place for accommodation in {to_place}, based on the number of adults {number_of_adults}, number of kids {number_of_kids}, {accommodation_budget}, and {going_date} and {returning_date}. Research and compare suitable hotels, hostels, serviced apartments, vacation rentals, and other accommodation options. Evaluate them based on cost, location, amenities, accessibility, safety, and suitability for the travel group. Research about all relevant options and compare which is the best option for the trip."""
                  ),
    expected_output = """Accommodation Analysis
    1. Accommodation options considered
    - List all the accommodation options you evaluated

    2. Recommended option
    - Name
    - Type
    - Distance from nearest airport/railway, if relevant.
    - Major tourist attractions nearby.
    - Cost per night
    - Total number of nights

    3. Amenities
    - List down all the available important amenities 

    4. Cost analysis
    - List down the total cost incurred in the whole stay.

    5. Reeason for your recommendation
    - Why will you recommend this over other alternatives?
    
    """,
    agent = accommodation_researcher
)

In [ ]:
activities_search = Task(
    description="""
    Research activities and tourists attractions in {to_place}. Consider the trip dates from {going_date} to {returning_date}, {number_of_adults},
    {number_of_kids}, and overall suitability for the group. Research about the ,cost relevamt and suitable activities using the appropriate search tool and compare available options and recommend the most suitable experiences.
    Keep in my the {activities_budget}.
    """,

    expected_output="""
    Activities Analysis

    1. Attractions Considered
    - List all the activities you have considered in your assessment

    2. Recommended Attractions
    - List all the activities you feel are the best, given all the parameters

    3. Estimated Costs
    - Give the estimated cost for all the activites recommended

    4. Recommended Food Experiences
    - Give out a list of all the delicacies available to try out in that area

    5. Reasons for Recommendation
    - Why you recommended only these options over others?
    """,

    agent=activities_researcher
)

In [51]:
itinerary_maker = Task(
    description="""
    Create a complete itinerary using transportation, accommodation and activities recommendations.
    Ensure the following: The tourists going on the trip should get the maximum of the place, and the whole trip should be balanced, not too much focus on one thing. Aim for a holistic experience for the tourists.
    """,
    expected_output="""
    Trip Summary

    Create a day by day itinerary plan for the whole trip. What to do in each day, how much time to spend doing a particular activity.
    
    Transportation Details
    - Transportation details
    
    Accommodation Details
    - Accommodation details
    
    Total Estimated Cost
    - Write the total cost of the trip, including every expenditure
    
    Additional Notes
    - Write some things to keep in mind, which might be important.
    """,

    context=[
        transportation_search,
        accommodation_search,
        activities_search
    ],

    agent=itinerary_researcher
)

In [52]:
qa_task = Task(
    description="""
    Review the complete travel plan generated by the previous tasks.
    Carefully verify:

    1. Transportation Feasibility
       - Transportation recommendations are practical.
       - Travel durations are realistic.
       - Departure and arrival timings are reasonable.

    2. Accommodation Suitability
       - Accommodation is suitable for the number of adults and children.
       - Accommodation cost remains within the accommodation budget.
       - Location is convenient for the planned itinerary.

    3. Activities Suitability
       - Activities are appropriate for the travelers.
       - Activities fit within the available time.
       - Activities do not conflict with transportation schedules.

    4. Budget Verification
       - Transportation budget is not exceeded.
       - Accommodation budget is not exceeded.
       - Overall trip costs are reasonable.

    5. Itinerary Consistency
       - Dates are correct.
       - Day-by-day schedule is realistic.
       - No overlapping activities.
       - No impossible travel arrangements.

    Identify any problems, inconsistencies, missing information, or unrealistic recommendations.

    If issues are found, suggest specific improvements.
    If no issues are found, explain why the plan is satisfactory.
    """,

    expected_output="""
    Travel Plan Quality Assurance Report

    1. Transportation Review
       - Findings
       - Issues (if any)

    2. Accommodation Review
       - Findings
       - Issues (if any)

    3. Activities Review
       - Findings
       - Issues (if any)

    4. Budget Review
       - Transportation Budget Status
       - Accommodation Budget Status
       - Overall Budget Assessment

    5. Itinerary Review
       - Schedule Consistency
       - Timing Feasibility
       - Date Verification

    6. Problems Identified
       - List all issues found
       - If none, state "No significant issues found"

    7. Suggested Improvements
       - Recommendations for improving the travel plan

    8. Final Verdict
       - Approved / Approved with Suggestions / Needs Revision
       - Brief justification
    """,

    context=[
        transportation_search,
        accommodation_search,
        activities_search,
        itinerary_maker
    ],

    agent=qa_researcher
)

In [53]:
crew = Crew(
    agents=[transportation_researcher, accommodation_researcher, activities_researcher, itinerary_researcher, qa_researcher],
    tasks=[transportation_search, accommodation_search, activities_search, itinerary_maker, qa_task],
    verbose = True
)

In [56]:
import sys
print(sys.version)

3.13.5 (v3.13.5:6cb20a219a8, Jun 11 2025, 12:23:45) [Clang 16.0.0 (clang-1600.0.26.6)]


In [58]:
result = await crew.kickoff_async(
    inputs={
        "from_place": "YOUR_INPUT",
        "to_place": "YOUR_INPUT",
        "going_date": "INPUT_INPUT",
        "returning_date": "YOUR_INPUT",
        "number_of_adults": YOUR_INPUT,
        "number_of_kids": YOUR_INPUT,
        "transportation_budget": YOUR_INPUT,
        "accommodation_budget": YOUR_INPUT,
        "activities_budget": YOUR_INPUT
    }
)

print(result.raw)

NameError: name 'YOUR_INPUT' is not defined